# Stock Market Anomaly Detection


## Initial Setup


In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from src.models.isolation_forest import IsolationForest
from src.models.zscore import zscore
from src.models.dbscan import DBSCAN
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from itertools import product
import plotly.graph_objects as go
from src.utils.paths import PROJECT_ROOT, ARTIFACTS,HYPERPARAMS,DATA
import json


import warnings

warnings.filterwarnings("ignore")

## Configuration


In [6]:
timeframe = "1D"
stock_name = "API"

isolation_forest_features = ["volume","returns","volatility","bb_width"]
dbscan_features = ["volume","returns","volatility","bb_width"]
features = list({*isolation_forest_features, *dbscan_features})

## Data Loading and Filtering


In [7]:

all_days = []
start_date = datetime.strptime("2024-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2026-05-15", "%Y-%m-%d")

# start_date = date.today() - relativedelta(years=1)
# end_date = date.today()

df = pd.read_csv(DATA / f"{stock_name}.csv")
           
if isinstance(start_date, str):
    start_date = datetime.strptime(start_date, "%Y-%m-%d").date()

if isinstance(end_date, str):
    end_date = datetime.strptime(end_date, "%Y-%m-%d").date()

df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
df = df.sort_index()

df = df.loc[start_date:end_date]         

# drop unnecessary columns
df = df.drop(columns=["date"], errors="ignore")

## Data Preprocessing

### Resampling/Aggregation


In [8]:
df = df.resample(timeframe).agg(
    open=("close", "first"),
    high=("close", "max"),
    low=("close", "min"),
    close=("close", "last"),
    volume=("volume", "sum"),
)

### Data cleaning


In [9]:
# print(df.columns)

# drop unnecessary columns
df = df.drop(columns=["date"], errors="ignore")

# Filter dataframe
df = df.drop_duplicates()

## Feature Engineering


### Compute Base features


In [10]:
# Compute features
df["returns"] = df["close"].pct_change()
df["close"] = df["close"].replace(0, np.nan)
df["volatility"] = df["returns"].rolling(window=20).std()


print(df[["close", "volume", "returns", "volatility"]].isna().sum())

close          1
volume         0
returns        1
volatility    20
dtype: int64


In [11]:
# # Plot volatitlity and percent change

# fig, axes = plt.subplots(3, 1, figsize=(12, 15))
# axes[0].plot(df.index, df["close"])
# axes[0].set_title(f"{stock_name} prices")
# axes[1].plot(df.index, df["returns"])
# axes[1].set_title(f"{stock_name} returnss")
# axes[2].plot(df.index, df["volatility"])
# axes[2].set_title(f"{stock_name} 20-day volatility")
# plt.tight_layout()

# plt.show()

### Compute additional features

**SMA:** Simple moving average of last N Days

**RSI:** Relative strength index measures the speed and magnitude of recent price changes to evaluate overbought or oversold conditions  
Above 70 → asset may be overbought (close went up too fast, may pull back)  
Below 30 → asset may be oversold (close dropped too fast, may bounce back)  
Around 50 → neutral momentum

**Boilinger Bands:** It consists of a:

Middle band: a 20 day SMA  
Upper band: middle band + 2*standard deviation  
Lower band: middle band - 2*standard deviation

Wide bands->volatile market
Price near upperband -> overbought
Price near lowerband -> oversold

When price moves outside this band means theres a breakout


In [12]:
# Feature Enginneering


# 1. Calculate Simple moving averages of last N days

df["SMA_10"] = df["close"].rolling(window=10).mean()
df["SMA_20"] = df["close"].rolling(window=20).mean()
df["SMA_50"] = df["close"].rolling(window=50).mean()


# 2. Calculate Relative Strength Index
def calculate_rsi(data, periods=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))


df["RSI"] = calculate_rsi(df["close"])


# 3. Calculate Boilinger bands
df["Upper_BB"] = df["SMA_20"] + (df["close"].rolling(window=20).std() * 2)
df["Lower_BB"] = df["SMA_20"] - (df["close"].rolling(window=20).std() * 2)

df["bb_width"] = (df["Upper_BB"] - df["Lower_BB"]) / df["close"]


df[features] = df[features].apply(pd.to_numeric, errors="coerce")
# converts every value in features columns to numeric value
df.replace([np.inf, -np.inf], np.nan, inplace=True)
# replace inf and -inf

df = df.dropna(subset=features)  # drops rows where any of the features are NaN


df.tail()

,open,high,low,close,volume,returns,volatility,SMA_10,SMA_20,SMA_50,RSI,Upper_BB,Lower_BB,bb_width
date,,,,,,,,,,,,,,
2026-05-11,349.0,349.0,349.0,349.0,416462.0,0.035608,0.012431,336.37,341.000,338.664,54.444444,355.298914,326.701086,0.081942
2026-05-12,345.0,345.0,345.0,345.0,294922.0,-0.011461,0.012662,336.77,340.700,339.744,56.844548,354.352684,327.047316,0.079146
2026-05-13,343.8,343.8,343.8,343.8,104602.0,-0.003478,0.012568,337.24,340.480,340.664,53.301887,353.760552,327.199448,0.077257
2026-05-14,348.0,348.0,348.0,348.0,139575.0,0.012216,0.012346,338.45,340.195,341.684,58.901099,352.489025,327.900975,0.070655
2026-05-15,349.0,349.0,349.0,349.0,326556.0,0.002874,0.012084,339.86,340.170,342.624,59.784946,352.386141,327.953859,0.070007


### Plot technical Analysis


In [13]:
df_filtered = df.copy()

if(timeframe!="1D"):
    df_filtered = df_filtered.between_time("11:00", "15:00")


# Create the figure
fig = go.Figure()

# Add 'Close' close line
fig.add_trace(
    go.Scatter(
        x=df_filtered.index,
        y=df_filtered["close"],
        mode="lines",
        name="Close",
        line=dict(color="blue", width=2),
    )
)

# Add 'SMA 10' line
fig.add_trace(
    go.Scatter(
        x=df_filtered.index,
        y=df_filtered["SMA_10"],
        mode="lines",
        name="SMA 10",
        line=dict(color="green", width=2, dash="dot"),
    )
)

# Add 'SMA 50' line
fig.add_trace(
    go.Scatter(
        x=df_filtered.index,
        y=df_filtered["SMA_50"],
        mode="lines",
        name="SMA 50",
        line=dict(color="orange", width=2, dash="dash"),
    )
)

# Add 'Upper BB' line
fig.add_trace(
    go.Scatter(
        x=df_filtered.index,
        y=df_filtered["Upper_BB"],
        mode="lines",
        name="Upper BB",
        line=dict(color="red", width=1, dash="dot"),
    )
)

# Add 'Lower BB' line
fig.add_trace(
    go.Scatter(
        x=df_filtered.index,
        y=df_filtered["Lower_BB"],
        mode="lines",
        name="Lower BB",
        line=dict(color="purple", width=1, dash="dot"),
    )
)

# Update layout for a larger figure size and title
fig.update_layout(
    title=f"{stock_name} Stock close with Technical Indicators",
    xaxis_title="Date",
    yaxis_title="Stock close",
    legend=dict(
        x=0, y=1, bgcolor="rgba(255,255,255,0)", bordercolor="rgba(255,255,255,0)"
    ),
    autosize=False,
    width=1200,
    height=600,
)


if(timeframe != "1D"):
    fig.update_xaxes(
        rangebreaks=[
            dict(bounds=["sat", "sat"]),  # remove weekends
            dict(bounds=[15, 11], pattern="hour"),  # keep only 11–15
        ]
    )


# Show the plot
fig.show()

In [14]:
from src.utils.load import load_json

best_params = load_json(HYPERPARAMS / f"{stock_name}.json")[f"{timeframe}"]

z_score_threshold = best_params["z_score"]["threshold"]

n_estimators = best_params["isolation_forest"]["n_estimators"]
contamination = best_params["isolation_forest"]["contamination"]

epsilon = best_params["dbscan"]["eps"]
min_samples = best_params["dbscan"]["min_pts"]

## Anomaly Detection Techniques


### Z Score


In [15]:
# Calculate Z Score for close prices
df["Anomaly_Z_Score"] = zscore(df["close"])

# Identify anomalies where Z-Score >3 or < -3
anomalies_zscore = df[abs(df["Anomaly_Z_Score"]) > z_score_threshold]

# Create the figure
fig = go.Figure()

# Add 'Close' close line
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["close"],
        mode="lines",
        name="Close prices",
        line=dict(color="blue", width=2),
    )
)

# Add anomalies (Z-score > 3 or < -3) as red markers
fig.add_trace(
    go.Scatter(
        x=anomalies_zscore.index,
        y=anomalies_zscore["close"],
        mode="markers",
        name="Anomalies (Z-Score)",
        marker=dict(color="red", size=8, symbol="circle"),
        showlegend=True,
    )
)

# Update layout for a larger figure size and title
fig.update_layout(
    title=f"{stock_name} Stock prices with Anomalies Detected Using Z-Score",
    xaxis_title="Date",
    yaxis_title="Stock close",
    autosize=False,
    width=1200,
    height=600,
)

# Show the plot
fig.show()
# df.sort_values(by="Anomaly_Z_Score",ascending=False).head()
anomalies_zscore.head()

,open,high,low,close,volume,returns,volatility,SMA_10,SMA_20,SMA_50,RSI,Upper_BB,Lower_BB,bb_width,Anomaly_Z_Score
date,,,,,,,,,,,,,,,


### Standardization

To standardize means to convert the data to a uniform scale such that the data is normalized and the model can learn better
this is important because the model is sensitive to the scale of the data as returns maybe in the range of -0.2 to 0.2 but the other features are in the range of 1000 to 1000000 like quantity


**Standard Scaler**: uses mean and s.d. such that data has mean of 0 and s.d of 1. it is highly sensitive to outliers due to the use of mean. It is useful for algorithms that assume normal distribution. such as regression and KNN

X_scaled= (x − mean) / s.d

**Robust Scaler**: uses median and I.Q.R such that the median of the data points is 0. and the distance between Q1 and Q3 is 1 unit. Highly suitable for financial data as outliers are preserved. Also suitable for real world noisy datasets.

X_scaled =  x-median/q3-q1


**Min Max Scaler**: compresses everything into [0,1] where one extreme value becomes 0 and the other 1. Here the outliers dominate the scaling as it is very sensitive to extreme values. it is useful for features with known bounds such as RSI(1-100)

X_scaled = X - Xmin / Xmax - Xmin







RobustScaler was selected for feature normalization because financial market features such as returns, volatility, and trading volume frequently contain extreme values and non-Gaussian distributions. Unlike StandardScaler, which relies on the mean and standard deviation, RobustScaler uses the median and interquartile range (IQR), making it less sensitive to outliers. This is particularly important in anomaly detection systems because the anomalies themselves should not distort the scaling process. Robust scaling also improves the effectiveness of distance-based algorithms such as DBSCAN and ensures more stable feature magnitudes across heterogeneous financial indicators.



 

In [16]:
# Only take the selected features
X = df[features]

scaler = RobustScaler()
X = scaler.fit_transform(X)

print(np.isfinite(X).all())
print(pd.DataFrame(X).isna().sum())

X = pd.DataFrame(X, columns=features, index=df.index)

X_if = X[isolation_forest_features].to_numpy()
X_dbscan = X[dbscan_features].to_numpy()


True
0    0
1    0
2    0
3    0
dtype: int64


### Isolation Forest Approach

Anomalies are easier to isolate than normal points and
because anomalies are rare and far from the cluster, a random split can isolate them quickly.
Normal points are surrounded by many neighbors, so they require more splits

It takes 3 parameters:

**1. n_estimators:** 200 means 200 independent trees are built and then sees how quickly a given point gets isolated across these trees.  
**2. contamination:** 0.01 assumes that 1% of the data is anomalous  
**3. max_depth:** cap a trees height  
**4. random_state:** for reproducibility purposes as without it every run gives slightly different anomalies

Random seed → geneprice random splits  
 ↓  
Build 200 isolation trees  
 ↓  
Compute anomaly scores  
 ↓  
Use contamination (1%) to label anomalies

isolation forest converts the average path length to scores  
short path → high anomaly score  
long path → low anomaly score

max-depth = ceil(log2(max_samples))  
len(X) is how many training rows you have.  
np.log2(n) is “how many times can you halve n before you get down to ~1” — a standard rough size for depth in random/binary tree pictures.  
np.ceil(...) rounds up so you always get an integer depth.  
int(...) makes it a plain Python int for your IsolationForest.

if training data becomes huge we instead find depth by choosing a max-sample size to randomly select samples to cap a tree's height


In [17]:
# We can also train the model and fit it in test data, But for simplicity purposes we will fit the model on the entire dataset.

# split_idx = int(len(df) * 0.8) #splits the dataframe into 80% train and 20% test
# df_train = df.iloc[:split_idx].copy() #creates a copy of the train dataframe from the first 80% of the dataframe
# df_test = df.iloc[split_idx:].copy() #creates a copy of the test dataframe from the last 20% of the dataframe


model_if = IsolationForest(
    n_trees=n_estimators, contamination=contamination, random_state=42
)

df["Anomaly_Isolation_Forest"] = model_if.fit_predict(X_if)

anomalies_if = df[df["Anomaly_Isolation_Forest"] == -1]


anomalies_if.head()

,open,high,low,close,volume,returns,volatility,SMA_10,SMA_20,SMA_50,RSI,Upper_BB,Lower_BB,bb_width,Anomaly_Z_Score,Anomaly_Isolation_Forest
date,,,,,,,,,,,,,,,,
2024-03-04,193.3,193.3,193.3,193.3,23832.0,0.099545,0.026954,182.25,190.345,NaN,41.067762,209.440271,171.249729,0.197571,-1.456701,-1
2024-08-06,250.1,250.1,250.1,250.1,1228937.0,-0.072673,0.040133,243.77,219.725,190.182,69.362512,278.205546,161.244454,0.467657,-0.400211,-1
2024-08-14,346.3,346.3,346.3,346.3,2237172.0,0.099714,0.044827,285.49,255.490,206.160,83.579545,333.544310,177.435690,0.450790,1.389127,-1
2024-08-15,375.0,375.0,375.0,375.0,1734493.0,0.082876,0.045965,297.18,263.800,210.282,85.854136,355.176399,172.423601,0.487341,1.922952,-1
2024-08-18,367.0,367.0,367.0,367.0,2142354.0,-0.021333,0.047123,308.49,271.650,214.282,82.165297,370.255188,173.044812,0.537358,1.774151,-1


In [18]:
fig = go.Figure()

# Add Close prices line
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["close"],
        mode="lines",
        name="Close prices",
        line=dict(color="blue", width=2),
    )
)

# Add anomalies (Isolation Forest) as red markers
fig.add_trace(
    go.Scatter(
        x=anomalies_if.index,
        y=anomalies_if["close"],
        mode="markers",
        name="Anomalies (Isolation Forest)",
        marker=dict(color="red", size=8, symbol="circle"),
        showlegend=True,
    )
)

# Update layout for a larger figure size and title
fig.update_layout(
    title=f"{stock_name} Stock prices with Anomalies Detected Using Isolation Forest",
    xaxis_title="Date",
    yaxis_title="Stock close",
    autosize=False,
    width=1200,
    height=600,
)

# Show the plot
fig.show()

### DBSCAN

DBSCAN finds anomalies by clustering the data points into density based clusters. It has 2 parameters

**eps:** It is the radius of the neighborhood around a data point, i.e the max-distance between two samples for them to be considered neighbors  
**min_pts:** The minimum number of data points required within the eps radius to form a dense region.


In [19]:
dbscan = DBSCAN(eps=epsilon, min_pts=min_samples)
df["Anomaly_DBSCAN"] = dbscan.fit_predict(X_dbscan)

anomalies_dbscan = df.copy()
anomalies_dbscan["Color"] = anomalies_dbscan["Anomaly_DBSCAN"].apply(
    lambda x: "red" if x == -1 else "blue"
)
anomalies_dbscan.head()

,open,high,low,close,volume,returns,volatility,SMA_10,SMA_20,SMA_50,RSI,Upper_BB,Lower_BB,bb_width,Anomaly_Z_Score,Anomaly_Isolation_Forest,Anomaly_DBSCAN,Color
date,,,,,,,,,,,,,,,,,,
2024-02-04,197.5,197.5,197.5,197.5,185914.0,-0.006039,0.024208,207.13,199.380,NaN,52.951096,220.252404,178.507596,0.211366,-1.378580,1,1,blue
2024-02-05,199.0,199.0,199.0,199.0,136985.0,0.007595,0.024203,205.98,200.105,NaN,48.237477,219.773781,180.436219,0.197676,-1.350680,1,1,blue
2024-02-06,202.0,202.0,202.0,202.0,176832.0,0.015075,0.024284,204.38,200.880,NaN,58.585859,219.484312,182.275688,0.184201,-1.294879,1,1,blue
2024-02-07,203.4,203.4,203.4,203.4,134579.0,0.006931,0.024234,203.62,201.750,NaN,55.818966,219.002734,184.497266,0.169643,-1.268839,1,1,blue
2024-02-08,201.0,201.0,201.0,201.0,386327.0,-0.011799,0.024031,202.82,202.655,NaN,45.215311,217.471275,187.838725,0.147426,-1.313479,1,1,blue


## Visualization


In [20]:
import plotly.express as px

fig = px.scatter(
    anomalies_dbscan,
    x="close",
    y="volume",
    color="Color",
    title=f"DBSCAN Clustering Results on {stock_name} Data",
    labels={"Color": "Cluster"},
    hover_data=["returns", "volatility"],  # Add more data to hover
)

# Update layout for better visualization
fig.update_layout(
    xaxis_title="Close prices",
    yaxis_title="volume",
    legend_title="Cluster",
    autosize=False,
    width=1200,
    height=600,
)

# Show the plot
fig.show()

## Evaluation


### Ensemble

There are 4 types of data points.
**1. Critical Anomaly:** When both dbscan and isolation forest consider a point to be anomalous
**2. Density Based Anomaly:** When only dbscan says a point is an anomaly
**3. Structure Based Anomaly:** When only isolation forest says a point is an anomaly
**4. Normal:** When both models say a model is not anomalous


We only use isolation forest and dbscan in visualizing and use z-score for setting a baseline for visualization


In [21]:
# Ensemble classification
# We can also score it using z score and results of the models but for simplicity we will just use the results of the models to classify the anomalies into 3 categories:
def classify_ensemble(row):

    if row["Anomaly_Isolation_Forest"] == -1 and row["Anomaly_DBSCAN"] == -1:
        return "Critical Anomaly"

    elif row["Anomaly_DBSCAN"] == -1:
        return "Density Based Anomaly"

    elif row["Anomaly_Isolation_Forest"] == -1:
        return "Structure Based Anomaly"

    return "Normal"

def z_label(z):
    if abs(z) < 1:
        return "Normal (within 1σ)"
    elif abs(z) < 2:
        return "Mild deviation (1–2σ)"
    elif abs(z) < 3:
        return "Strong deviation (2–3σ)"
    else:
        return "Extreme deviation (>3σ)"

df["ensemble_label"] = df.apply(classify_ensemble, axis=1)
df["z_label"] = df["Anomaly_Z_Score"].apply(z_label)

critical = df[df["ensemble_label"] == "Critical Anomaly"]
density = df[df["ensemble_label"] == "Density Based Anomaly"]
structure = df[df["ensemble_label"] == "Structure Based Anomaly"]

fig = go.Figure()

# Add the line for the closing close
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["close"],
        mode="lines",
        name="Close prices",
        line=dict(color="blue"),
    )
)

# Critical anomalies
fig.add_trace(
    go.Scatter(
        x=critical.index,
        y=critical["close"],
        mode="markers",
        name="Critical",
        hovertemplate=
        "Time: %{x}<br>" +
        "Price: %{y}<br>" +
        "Deviation: %{customdata[0]:.2f}σ<br>" +
        "<extra></extra>",
        customdata=np.stack([
            critical["Anomaly_Z_Score"]
        ], axis=-1),
        marker=dict(color="red", size=10)
    )
)

# Density anomalies
fig.add_trace(
    go.Scatter(
        x=density.index,
        y=density["close"],
        mode="markers",
        name="Density",
        hovertemplate=
        "Time: %{x}<br>" +
        "Price: %{y}<br>" +
        "Deviation: %{customdata[0]:.2f}σ<br>" +
        "<extra></extra>",
        customdata=np.stack([
            density["Anomaly_Z_Score"]
        ], axis=-1),
        marker=dict(color="orange", size=8)
    )
)

# Structure anomalies
fig.add_trace(
    go.Scatter(
        x=structure.index,
        y=structure["close"],
        mode="markers",
        name="Structure",
        hovertemplate=
        "Time: %{x}<br>" +
        "Price: %{y}<br>" +
        "Deviation: %{customdata[0]:.2f}σ<br>" +
        "<extra></extra>",
        customdata=np.stack([
            structure["Anomaly_Z_Score"]
        ], axis=-1),
        marker=dict(color="green", size=8)
    )
)

# Update layout
fig.update_layout(
    title=f"{stock_name} Stock close with Detected Anomalies",
    xaxis_title="Date",
    yaxis_title="Close prices",
    legend_title="Legend",
    width=1200,
    height=600,
)

# Show the plot
fig.show()

In [22]:
print("Number of Z-Score Anomalies:", len(anomalies_zscore))
print(
    "Number of Isolation Forest Anomalies:",
    (df["Anomaly_Isolation_Forest"] == -1).sum(),
)
print("Number of DBSCAN Anomalies:", sum(anomalies_dbscan["Anomaly_DBSCAN"] == -1))

Number of Z-Score Anomalies: 0
Number of Isolation Forest Anomalies: 11
Number of DBSCAN Anomalies: 3


## Results

**1. Precision:** When the model says its an anomaly, how often it is correct? (false positives)  
High Precision->Fewer false alarms  
Precision= TP/(TP+FP)

**2. Recall:** Out of all real anomalies. how many did we catch? (false negatives)  
High recall -> less anomalies missed
Recall = TP/(TP+FN)

**3. F1-Score:** Balance between False positives and false negatives
Harmonic mean of precision and recall  
f1-score = 2*precision*recall/(precision+recall
)


In [23]:
# import seaborn as sns


# methods = ["Z_Score", "Isolation_Forest", "DBSCAN"]
# results = []

# for method in methods:
#     if method == "Z_Score":
#         predictions = (abs(df["Anomaly_Z_Score"]) > z_score_threshold).astype(int)
#     elif method == "Isolation_Forest":
#         predictions = (df["Anomaly_Isolation_Forest"] == -1).astype(int)
#     elif method == "DBSCAN":
#         predictions = (df["Anomaly_DBSCAN"] == -1).astype(int)

#     precision = precision_score(df["True_Anomaly"], predictions)
#     recall = recall_score(df["True_Anomaly"], predictions)
#     f1 = f1_score(df["True_Anomaly"], predictions)
#     # print(classification_report(df["True_Anomaly"], predictions));
#     cm = confusion_matrix(df["True_Anomaly"], predictions)

#     print(f"\nConfusion Matrix for {method}:")
#     print(cm)

#     plt.figure(figsize=(6, 5))
#     sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

#     plt.title(f"Confusion Matrix - {method}")
#     plt.xlabel("Predicted")
#     plt.ylabel("True")

#     plt.show()

#     results.append(
#         {"Method": method, "Precision": precision, "Recall": recall, "F1-Score": f1}
#     )

# results_df = pd.DataFrame(results)
# print("\nComparison of all methods:")
# print(results_df)